# 01. Cluster QC Review

**목적**: Snakemake 파이프라인 완료 후 클러스터링 결과를 검토하고, 저품질 클러스터를 식별/제거한다.

**입력**: `output/checkpoints/{dataset}/07_clustered.h5ad`  
**출력**: `output/checkpoints/{dataset}/07_clustered_clean.h5ad` (저품질 클러스터 제거)


In [ ]:
import sys
sys.path.insert(0, "../../")  # 프로젝트 루트
sys.path.insert(0, "../")     # src/

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from modules.io import load_adata, save_adata
from modules.qc import summarize_qc, plot_qc_violin
from modules.plotting import umap_panel

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white")

In [ ]:
# ============================================================
# 설정 — 분석할 데이터셋 선택
# ============================================================
DATASET = "human_genes_only"  # samples.yaml의 dataset 이름
CHECKPOINT_DIR = f"../../output/checkpoints/{DATASET}"

adata = load_adata(f"{CHECKPOINT_DIR}/07_clustered.h5ad")
print(adata)

## 1. QC 지표 분포 확인

In [ ]:
# QC 지표를 클러스터별로 시각화
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="leiden",
    jitter=False,
    rotation=45,
)

In [ ]:
# UMAP — QC 지표로 색상 표시
umap_panel(
    adata,
    color=["leiden", "n_genes_by_counts", "total_counts", "pct_counts_mt"],
    ncols=4,
)

## 2. 클러스터별 QC 통계

In [ ]:
# 클러스터별 QC 요약
qc_cols = [c for c in ["n_genes_by_counts", "total_counts", "pct_counts_mt", "doublet_score"] 
           if c in adata.obs.columns]
cluster_qc = adata.obs.groupby("leiden")[qc_cols].median().sort_index()
cluster_qc["n_cells"] = adata.obs.groupby("leiden").size()
cluster_qc

## 3. 저품질 클러스터 제거

아래 `LOW_QUALITY_CLUSTERS` 리스트에 제거할 클러스터 번호를 입력하세요.  
기준: 높은 mt% (>20%), 낮은 n_genes (<200), 높은 doublet_score 등

In [ ]:
# ============================================================
# 수동 설정: 제거할 클러스터 (위 통계 확인 후 입력)
# ============================================================
LOW_QUALITY_CLUSTERS = []  # 예: ["12", "25", "38"]

if LOW_QUALITY_CLUSTERS:
    mask = ~adata.obs["leiden"].isin(LOW_QUALITY_CLUSTERS)
    adata_clean = adata[mask].copy()
    print(f"Removed {(~mask).sum()} cells from clusters {LOW_QUALITY_CLUSTERS}")
    print(f"Remaining: {adata_clean.n_obs} cells")
else:
    adata_clean = adata.copy()
    print("No clusters removed.")

In [ ]:
# 정리 후 UMAP 확인
sc.pl.umap(adata_clean, color="leiden", legend_loc="on data", title="After QC filtering")

In [ ]:
# 저장
out_path = f"{CHECKPOINT_DIR}/07_clustered_clean.h5ad"
save_adata(adata_clean, out_path)
print(f"Saved: {out_path}")